# End-to-End Hierarchical Cell Type Classification with HCE Loss (CORRECTED)

This notebook trains a C2S encoder + classification head end-to-end using Hierarchical Cross-Entropy (HCE) loss on the balanced lung dataset.

**Key approach:**
- C2S encoder + learnable classification head
- End-to-end training with HCE loss
- Balanced dataset (capped max cells per type)
- Hierarchy incorporated directly into training via reachability matrix

**CORRECTIONS in this version:**
- ✅ Fixed reachability matrix directionality (parent→child, not child→parent)
- ✅ Added transitive closure computation (Floyd-Warshall algorithm)
- ✅ Corrected HCE loss to give credit for ancestor predictions

## 1. Import Required Libraries

In [4]:
import os
import sys
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add cell2sentence to path
sys.path.insert(0, '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/src')

from cell2sentence.hce_trainer import build_reachability_matrix_from_ontology

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Configuration

In [5]:
# File paths
LUNG_H5AD_PATH = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad'
OUT_DIR = '/home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_end_to_end_results_corrected'
os.makedirs(OUT_DIR, exist_ok=True)

# Model configuration
C2S_MODEL_NAME = 'vandijklab/C2S-Pythia-410m-cell-type-conditioned-cell-generation'
TOP_K_GENES = 100

# Dataset balancing parameters
MAX_CELLS_PER_TYPE = 1000
MIN_CELLS_PER_TYPE = 20
ANN_COL = 'ann_finest_level'

# Training parameters
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
N_EPOCHS = 10
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

# Evaluation split
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Device configuration
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        device = torch.device("mps")
        print("Using Apple Metal Performance Shaders (MPS)")
    else:
        device = torch.device("cpu")
        print("Using CPU (no GPU detected)")
    return device

device = get_device()

print("\n" + "="*80)
print("CONFIGURATION SUMMARY")
print("="*80)
print(f"Dataset: {LUNG_H5AD_PATH}")
print(f"Output: {OUT_DIR}")
print(f"Model: {C2S_MODEL_NAME.split('/')[-1]}")
print(f"Device: {device}")
print(f"\nDataset Balancing:")
print(f"  Max cells per type: {MAX_CELLS_PER_TYPE}")
print(f"  Min cells per type: {MIN_CELLS_PER_TYPE}")
print(f"\nTraining Config:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Top K genes: {TOP_K_GENES}")

Using CUDA GPU: NVIDIA GeForce RTX 5060 Ti

CONFIGURATION SUMMARY
Dataset: /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad
Output: /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_end_to_end_results_corrected
Model: C2S-Pythia-410m-cell-type-conditioned-cell-generation
Device: cuda

Dataset Balancing:
  Max cells per type: 1000
  Min cells per type: 20

Training Config:
  Batch size: 8
  Learning rate: 0.0001
  Epochs: 10
  Top K genes: 100


## 3. Load and Balance Dataset

In [6]:
print("\n" + "="*80)
print("LOADING AND BALANCING DATASET")
print("="*80)

# Load dataset
print(f"\n📚 Loading {LUNG_H5AD_PATH}...")
adata = sc.read_h5ad(LUNG_H5AD_PATH, backed='r')
print(f"   Original shape: {adata.shape}")

# Check for annotation column
if ANN_COL not in adata.obs.columns:
    ann_cols = [col for col in adata.obs.columns if 'cell' in col.lower() or 'type' in col.lower()]
    if ann_cols:
        ANN_COL = ann_cols[0]
        print(f"⚠️  '{ANN_COL}' not found. Using '{ANN_COL}' instead")
    else:
        raise ValueError(f"Could not find annotation column")

# Show original distribution
print(f"\n📊 Original class distribution (top 10):")
orig_counts = adata.obs[ANN_COL].value_counts().sort_values(ascending=False)
for idx, (cell_type, count) in enumerate(orig_counts.head(10).items(), 1):
    pct = count / len(adata.obs) * 100
    print(f"   {idx:2d}. {cell_type:40s}: {count:5d} cells ({pct:5.1f}%)")
if len(orig_counts) > 10:
    print(f"   ... and {len(orig_counts) - 10} more cell types")

# Balance the dataset
print(f"\n🎯 Balancing dataset (capping at {MAX_CELLS_PER_TYPE} per type)...")
labels = adata.obs[ANN_COL].astype(str)
counts = labels.value_counts()

keep_labels = counts[counts >= MIN_CELLS_PER_TYPE].index.tolist()
print(f"   Keeping {len(keep_labels)} classes with >= {MIN_CELLS_PER_TYPE} cells")
print(f"   Dropping {len(counts) - len(keep_labels)} classes (too rare)")

# Sample up to MAX_CELLS_PER_TYPE from each label
sampled_idx = []
for lbl in keep_labels:
    idxs = np.where(labels.values == lbl)[0]
    n_keep = min(len(idxs), MAX_CELLS_PER_TYPE)
    chosen = np.random.choice(idxs, size=n_keep, replace=False)
    sampled_idx.extend(chosen.tolist())

# Create balanced AnnData
sampled_idx = np.array(sampled_idx, dtype=int)
sampled_idx = np.sort(sampled_idx)
adata_balanced = adata[sampled_idx].to_memory()

# Remove unknown/invalid annotations
print(f"\n🧹 Removing unknown/invalid annotations...")
unknown_terms = ['unknown', 'Unknown', 'UNKNOWN', 'nan', 'NA', 'N/A', 'none', 'None']
valid_mask = ~adata_balanced.obs[ANN_COL].isin(unknown_terms)
print(f"   Found {(~valid_mask).sum()} unknown cells")
adata_balanced = adata_balanced[valid_mask].copy()

print(f"\n✅ Balanced dataset created (after removing unknowns):\"")
print(f"   Total cells: {adata_balanced.n_obs}")
print(f"   Cell types: {len(adata_balanced.obs[ANN_COL].unique())}\"")

# Show new distribution
new_counts = adata_balanced.obs[ANN_COL].value_counts().sort_values(ascending=False)
print(f"\n📊 Balanced class distribution (top 10):\"")
for idx, (cell_type, count) in enumerate(new_counts.head(10).items(), 1):
    pct = count / len(adata_balanced.obs) * 100
    print(f"   {idx:2d}. {cell_type:40s}: {count:5d} cells ({pct:5.1f}%)")
if len(new_counts) > 10:
    print(f"   ... and {len(new_counts) - 10} more cell types")

# Save counts
orig_counts.to_csv(os.path.join(OUT_DIR, 'original_counts.csv'), header=['count'])
new_counts.to_csv(os.path.join(OUT_DIR, 'balanced_counts.csv'), header=['count'])
print(f"\n✅ Saved dataset counts")


LOADING AND BALANCING DATASET

📚 Loading /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung.h5ad...
   Original shape: (2282447, 55329)

📊 Original class distribution (top 10):
    1. Unknown                                 : 623313 cells ( 27.3%)
    2. Alveolar macrophages                    : 258823 cells ( 11.3%)
    3. AT2                                     : 155346 cells (  6.8%)
    4. CD8 T cells                             : 113641 cells (  5.0%)
    5. Multiciliated (non-nasal)               : 113074 cells (  5.0%)
    6. Monocyte-derived Mph                    : 88248 cells (  3.9%)
    7. CD4 T cells                             : 82698 cells (  3.6%)
    8. Goblet (nasal)                          : 72359 cells (  3.2%)
    9. Classical monocytes                     : 64980 cells (  2.8%)
   10. Suprabasal                              : 60997 cells (  2.7%)
   ... and 52 more cell types

🎯 Balancing dataset (capping at 1000 per type)...
   Keeping 62 

## 4. Build Cell Type Hierarchy

In [ ]:
print("\n" + "="*80)
print("BUILDING CELL TYPE HIERARCHY")
print("="*80)

# Find annotation level columns (HLCA standard)
level_columns = sorted([col for col in adata_balanced.obs.columns 
                        if col.startswith('ann_level_') and col[-1].isdigit()])

print(f"\n🔍 Found {len(level_columns)} hierarchical annotation levels:")
for col in level_columns:
    n_unique = adata_balanced.obs[col].nunique()
    n_not_na = adata_balanced.obs[col].notna().sum()
    print(f"  - {col}: {n_unique} unique values ({n_not_na}/{len(adata_balanced.obs)} cells annotated)")

# Verify annotation hierarchy structure
print(f"\n📋 Verifying hierarchy structure (sample cell annotations):")
sample_idx = 0
for col in level_columns:
    val = adata_balanced.obs.iloc[sample_idx][col]
    print(f"  {col}: {val}")

if len(level_columns) < 2:
    print("\n⚠️  WARNING: Could not find hierarchical annotation levels.")
    ontology_dict = {ct: None for ct in adata_balanced.obs[ANN_COL].unique()}
    all_cell_types = list(adata_balanced.obs[ANN_COL].unique())
else:
    # Build hierarchy from annotation levels
    print(f"\n🏗️  Building hierarchy from {len(level_columns)} annotation levels...")
    
    all_cell_types = []
    level_to_types = {}
    
    for level_col in level_columns:
        unique_types = [t for t in adata_balanced.obs[level_col].dropna().unique() 
                       if t not in ['unknown', 'Unknown', 'nan', 'NA', 'None']]
        level_to_types[level_col] = unique_types
        all_cell_types.extend([t for t in unique_types if t not in all_cell_types])
    
    # Build parent-child relationships
    ontology_dict = {}
    
    for idx in range(len(adata_balanced.obs)):
        for j in range(len(level_columns) - 1):
            child_col = level_columns[j + 1]
            parent_col = level_columns[j]
            
            child = adata_balanced.obs.iloc[idx][child_col]
            parent = adata_balanced.obs.iloc[idx][parent_col]
            
            if (pd.notna(child) and pd.notna(parent) and 
                child not in ['unknown', 'Unknown', 'None'] and 
                parent not in ['unknown', 'Unknown', 'None'] and
                str(child) != 'nan' and str(parent) != 'nan'):
                
                if child not in ontology_dict:
                    ontology_dict[child] = parent
    
    # Add root nodes
    if level_columns:
        root_col = level_columns[0]
        for root_type in level_to_types[root_col]:
            if root_type not in ontology_dict:
                ontology_dict[root_type] = None
    
    print(f"\n✅ Built hierarchy with:")
    print(f"   Total cell types: {len(all_cell_types)}")
    print(f"   Parent-child relationships: {len([v for v in ontology_dict.values() if v is not None])}")
    
    # Show example paths with verification
    print(f"\n📊 Example hierarchy paths:")
    example_count = 0
    invalid_paths = 0
    for child, parent in ontology_dict.items():
        if parent is not None and example_count < 5:
            # Verify the relationship is valid
            if str(parent) == 'None' or str(parent) == 'nan':
                print(f"   ⚠️  {child} → {parent} (INVALID - parent should not be None/nan)")
                invalid_paths += 1
            else:
                print(f"   {child} → {parent}")
            example_count += 1
    
    if invalid_paths > 0:
        print(f"\n⚠️  WARNING: Found {invalid_paths} invalid hierarchy paths!")
        print(f"   This may indicate reversed annotation levels or data quality issues.")

# Save ontology
ontology_df = pd.DataFrame(list(ontology_dict.items()), columns=['child', 'parent'])
ontology_df.to_csv(os.path.join(OUT_DIR, 'ontology.csv'), index=False)
print(f"\n✅ Saved ontology to {os.path.join(OUT_DIR, 'ontology.csv')}")


BUILDING CELL TYPE HIERARCHY

🔍 Found 5 hierarchical annotation levels:
  - ann_level_1: 4 unique values (57611/57611 cells annotated)
  - ann_level_2: 11 unique values (57611/57611 cells annotated)
  - ann_level_3: 25 unique values (55902/57611 cells annotated)
  - ann_level_4: 39 unique values (50529/57611 cells annotated)
  - ann_level_5: 17 unique values (30381/57611 cells annotated)

🏗️  Building hierarchy from 5 annotation levels...

✅ Built hierarchy with:
   Total cell types: 94
   Parent-child relationships: 90

📊 Example hierarchy paths:
   Blood vessels → Endothelial
   EC arterial → Blood vessels
   None → EC arterial
   Lymphoid → Immune
   T cell lineage → Lymphoid

✅ Saved ontology to /home/hugolab/Documents/Cell2Sentence-HugoLab/Cell2Sentence-HugoLab/lung_hce_end_to_end_results_corrected/ontology.csv


## 5. Prepare Cell Text Data

In [ ]:
print("\n" + "="*80)
print("PREPARING CELL TEXT DATA")
print("="*80)

# Helper functions
def cell_to_text(cell_vector, gene_names, top_k=100):
    """Convert cell expression to top-k gene names."""
    if hasattr(cell_vector, "toarray"):
        vec = cell_vector.toarray().flatten()
    else:
        vec = np.asarray(cell_vector).flatten()
    
    if vec.size == 0:
        return ""
    
    top_idx = np.argsort(vec)[-top_k:][::-1]
    top_genes = [str(gene_names[i]) for i in top_idx if gene_names[i]]
    return " ".join(top_genes)

# Convert cells to text
print(f"\n📝 Converting {adata_balanced.n_obs} cells to text...")
cell_texts = []
for i in range(adata_balanced.n_obs):
    text = cell_to_text(adata_balanced.X[i], adata_balanced.var_names, top_k=TOP_K_GENES)
    cell_texts.append(text)

# Filter empty
keep_idx = [i for i, t in enumerate(cell_texts) if isinstance(t, str) and t.strip() != ""]
cell_texts = [cell_texts[i] for i in keep_idx]
cell_ids = adata_balanced.obs_names[keep_idx].tolist()
cell_labels = adata_balanced.obs[ANN_COL].iloc[keep_idx].astype(str).values

print(f"✅ Converted {len(cell_texts)} cells to text")
print(f"   Sample: {cell_texts[0][:100]}...")


PREPARING CELL TEXT DATA

📝 Converting 57611 cells to text...


## 6. Create Dataset and Model

In [ ]:
# Load tokenizer
print(f"\n🔄 Loading Cell2Sentence tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(C2S_MODEL_NAME)
print(f"✅ Tokenizer loaded")

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(cell_labels)

print(f"\n✅ Label encoding complete:")
print(f"   Total cells: {len(labels_encoded)}")
print(f"   Total classes: {len(le.classes_)}")
print(f"   Classes: {le.classes_[:5]}...")

# Custom Dataset
class CellTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Split data
idx = np.arange(len(labels_encoded))
idx_trainval, idx_test, y_trainval, y_test = train_test_split(
    idx, labels_encoded, test_size=TEST_FRAC, stratify=labels_encoded, random_state=RANDOM_SEED
)
relative_val_frac = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
idx_train, idx_val, y_train, y_val = train_test_split(
    idx_trainval, y_trainval, test_size=relative_val_frac, stratify=y_trainval, random_state=RANDOM_SEED
)

print(f"\n✅ Data split complete:")
print(f"   Train: {len(idx_train)} cells")
print(f"   Val: {len(idx_val)} cells")
print(f"   Test: {len(idx_test)} cells")

# Create datasets
train_texts = [cell_texts[i] for i in idx_train]
val_texts = [cell_texts[i] for i in idx_val]
test_texts = [cell_texts[i] for i in idx_test]

train_dataset = CellTextDataset(train_texts, y_train, tokenizer)
val_dataset = CellTextDataset(val_texts, y_val, tokenizer)
test_dataset = CellTextDataset(test_texts, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✅ DataLoaders created")

## 7. Define Model Architecture

In [ ]:
# Load C2S encoder
print(f"\n🔄 Loading C2S encoder...")
c2s_model = AutoModel.from_pretrained(C2S_MODEL_NAME)
print(f"✅ C2S encoder loaded")

# Get hidden size
hidden_size = c2s_model.config.hidden_size
print(f"   Hidden size: {hidden_size}")

# Classification head
class ClassificationHead(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.dense = nn.Linear(input_dim, 256)
        self.dense2 = nn.Linear(256, num_classes)
        self.activation = nn.ReLU()
    
    def forward(self, hidden_states):
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = self.activation(hidden_states)
        hidden_states = self.dropout(hidden_states)
        logits = self.dense2(hidden_states)
        return logits

# Combined model
class C2SClassifier(nn.Module):
    def __init__(self, encoder, num_classes, hidden_size):
        super().__init__()
        self.encoder = encoder
        self.classifier = ClassificationHead(hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        encoder_output = self.encoder(input_ids, attention_mask=attention_mask, output_hidden_states=False)
        # Mean pooling over tokens
        last_hidden = encoder_output.last_hidden_state  # (batch_size, seq_len, hidden_size)
        attention_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_hidden = torch.sum(last_hidden * attention_mask_expanded, 1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(1), min=1e-9)
        mean_hidden = sum_hidden / sum_mask  # (batch_size, hidden_size)
        
        logits = self.classifier(mean_hidden)
        return logits

model = C2SClassifier(c2s_model, len(le.classes_), hidden_size)
model.to(device)
print(f"\n✅ Model created and moved to {device}")

## 8. Build Reachability Matrix and HCE Loss (CORRECTED)

In [ ]:
# Build reachability matrix (CORRECTED VERSION)
print("\n" + "="*80)
print("BUILDING REACHABILITY MATRIX FOR HCE LOSS (CORRECTED)")
print("="*80)

def build_reachability_matrix_corrected(ontology_dict, class_names):
    """
    Build reachability matrix from ontology with CORRECT directionality.
    
    Key corrections:
    - Parent CAN REACH child (not the other way around)
    - Includes transitive closure (if A→B and B→C, then A→C)
    - Uses Floyd-Warshall algorithm for transitive closure
    
    The matrix is such that reachability[i, j] = 1 means class i can reach class j.
    This allows HCE to give credit for predicting a class or any of its ancestors.
    """
    n_classes = len(class_names)
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    
    # Start with identity (each class reaches itself)
    reachability = np.eye(n_classes, dtype=np.float32)
    
    # Add direct parent-child relationships (CORRECTED DIRECTION)
    for child, parent in ontology_dict.items():
        if parent is not None and child in class_to_idx and parent in class_to_idx:
            child_idx = class_to_idx[child]
            parent_idx = class_to_idx[parent]
            # CORRECTED: Parent CAN reach child
            reachability[parent_idx, child_idx] = 1.0
    
    # Add transitive closure using Floyd-Warshall algorithm
    # This ensures if A→B and B→C, then A→C is also in the matrix
    print(f"\n📊 Computing transitive closure with Floyd-Warshall...")
    for k in range(n_classes):
        for i in range(n_classes):
            for j in range(n_classes):
                if reachability[i, k] and reachability[k, j]:
                    reachability[i, j] = 1.0
    
    return torch.tensor(reachability, dtype=torch.float32, device=device)

reachability_matrix = build_reachability_matrix_corrected(ontology_dict, le.classes_.tolist())
print(f"\n✅ Reachability matrix built: {reachability_matrix.shape}")
print(f"   Non-zero entries (including diagonal): {torch.count_nonzero(reachability_matrix).item()}")
print(f"   Non-diagonal relations: {torch.count_nonzero(reachability_matrix).item() - len(le.classes_)}")

# HCE Loss (CORRECTED VERSION)
class HCELoss(nn.Module):
    """
    Hierarchical Cross-Entropy Loss (CORRECTED VERSION)
    
    Key corrections:
    - Uses correct matrix indexing: matrix[:, true_label] gives ancestors
    - Gives credit for predicting the true class OR any of its ancestors
    - Sums probabilities of all acceptable (ancestor) predictions
    """
    def __init__(self, reachability_matrix):
        super().__init__()
        self.reachability_matrix = reachability_matrix
    
    def forward(self, logits, labels):
        """
        Compute Hierarchical Cross-Entropy loss.
        Uses reachability matrix to give partial credit for ancestor predictions.
        
        Args:
            logits: Model predictions, shape (batch_size, num_classes)
            labels: True labels, shape (batch_size,)
        
        Returns:
            Scalar loss value
        """
        # Get probabilities for each class
        probs = F.softmax(logits, dim=1)  # (batch_size, num_classes)
        
        batch_size = labels.shape[0]
        loss = 0.0
        
        for i in range(batch_size):
            true_label = labels[i].item()
            # CORRECTED: Get which predictions are acceptable
            # reachability_matrix[:, true_label] gives all classes that can reach true_label
            # This includes the true class itself and all its ancestors
            acceptable = self.reachability_matrix[:, true_label]  # shape: (num_classes,)
            
            # Sum probabilities of acceptable predictions
            acceptable_prob = torch.sum(probs[i] * acceptable)
            
            # Negative log likelihood
            loss -= torch.log(acceptable_prob + 1e-10)
        
        return loss / batch_size

hce_criterion = HCELoss(reachability_matrix)
print(f"\n✅ HCE Loss function created with corrected formula")

## 9. Training Loop

In [ ]:
print("\n" + "="*80)
print("TRAINING WITH CORRECTED HCE LOSS")
print("="*80)

# Optimizer - only optimize encoder and classifier, not fixing C2S weights
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * N_EPOCHS
warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_STEPS)
main_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS)

def get_scheduler(current_step):
    if current_step < WARMUP_STEPS:
        return warmup_scheduler
    else:
        return main_scheduler

# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    
    for batch in tqdm(loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    
    accuracy = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), accuracy, all_preds, all_labels

# Training loop
train_losses = []
val_losses = []
val_accs = []

print(f"\n📊 Starting training for {N_EPOCHS} epochs...\n")

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{N_EPOCHS}")
    
    # Training
    train_loss = train_epoch(model, train_loader, hce_criterion, optimizer, device)
    train_losses.append(train_loss)
    
    # Validation
    val_loss, val_acc, _, _ = evaluate(model, val_loader, hce_criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracy: {val_acc:.4f}")

print(f"\n✅ Training complete!")
print(f"   Final Val Accuracy: {val_accs[-1]:.4f}")

## 10. Evaluation on Test Set

In [ ]:
print("\n" + "="*80)
print("TEST SET EVALUATION")
print("="*80)

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, hce_criterion, device)

print(f"\n✅ Test Results:")
print(f"   Test Loss: {test_loss:.4f}")
print(f"   Test Accuracy: {test_acc:.4f}")

# Create confusion matrix
cm = confusion_matrix(test_labels, test_preds, labels=np.arange(len(le.classes_)))
print(f"   Confusion Matrix: {cm.shape}")

# Per-class metrics
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, labels=np.arange(len(le.classes_))
)

per_class_metrics = pd.DataFrame({
    'cell_type': le.classes_,
    'precision': precision,
    'recall': recall,
    'f1': f1
}).sort_values('f1', ascending=False)

metrics_path = os.path.join(OUT_DIR, 'per_class_metrics.csv')
per_class_metrics.to_csv(metrics_path, index=False)

print(f"\n📊 Top 15 Cell Types by F1 Score:")
print(per_class_metrics[['cell_type', 'precision', 'recall', 'f1']].head(15).to_string(index=False))

print(f"\n✅ Saved per-class metrics to {metrics_path}")

## 11. Visualization

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
epochs = np.arange(1, N_EPOCHS + 1)
ax.plot(epochs, train_losses, 'o-', label='Train Loss', linewidth=2, markersize=8)
ax.plot(epochs, val_losses, 's-', label='Val Loss', linewidth=2, markersize=8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('HCE Loss During Training (CORRECTED)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs, val_accs, 'o-', label='Val Accuracy', linewidth=2, markersize=8, color='green')
ax.axhline(y=test_acc, color='red', linestyle='--', label=f'Test Accuracy: {test_acc:.4f}', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Validation & Test Accuracy', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim([0, 1.0])

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, 'training_curves.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved training curves to {fig_path}")
plt.show()

# Confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', aspect='auto')
ax.set_title('Confusion Matrix - HCE Trained Model (CORRECTED)\n(Normalized by True Label)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.colorbar(im, ax=ax, label='Normalized Count')

# Reduced labels for readability
tick_interval = max(1, len(le.classes_) // 15)
tick_positions = np.arange(0, len(le.classes_), tick_interval)
tick_labels = [le.classes_[i] for i in tick_positions]

ax.set_xticks(tick_positions)
ax.set_yticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(tick_labels, fontsize=8)

plt.tight_layout()
cm_path = os.path.join(OUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved confusion matrix to {cm_path}")
plt.show()

# Per-class F1
fig, ax = plt.subplots(figsize=(12, 10))

top_n = 15
top_metrics = per_class_metrics.head(top_n)
x = np.arange(len(top_metrics))
width = 0.25

ax.barh(x - width, top_metrics['precision'].values, width, label='Precision', alpha=0.8)
ax.barh(x, top_metrics['recall'].values, width, label='Recall', alpha=0.8)
ax.barh(x + width, top_metrics['f1'].values, width, label='F1', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(top_metrics['cell_type'].values, fontsize=10)
ax.set_xlabel('Score', fontsize=11)
ax.set_title(f'Top {top_n} Cell Types - Performance Metrics', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim([0, 1.0])

plt.tight_layout()
f1_path = os.path.join(OUT_DIR, 'per_class_metrics.png')
plt.savefig(f1_path, dpi=200, bbox_inches='tight')
print(f"✅ Saved per-class metrics plot to {f1_path}")
plt.show()

## 12. Summary Report

In [ ]:
print("\n" + "="*80)
print("END-TO-END HCE TRAINING - SUMMARY REPORT (CORRECTED)")
print("="*80)

summary_text = f"""
DATASET CONFIGURATION
=====================
Original dataset:       {orig_counts.sum():,} cells
Balanced dataset:       {len(labels_encoded):,} cells
Max cells per type:     {MAX_CELLS_PER_TYPE}
Min cells per type:     {MIN_CELLS_PER_TYPE}
Number of cell types:   {len(le.classes_)}

HIERARCHY INFORMATION
====================
Total cell types:       {len(all_cell_types)}
Parent-child relations: {len([v for v in ontology_dict.values() if v is not None])}
Reachability matrix:    {reachability_matrix.shape} (with transitive closure)
Non-diagonal entries:   {torch.count_nonzero(reachability_matrix).item() - len(le.classes_)}

MODEL CONFIGURATION
===================
Encoder:                {C2S_MODEL_NAME.split('/')[-1]}
Classification head:    Dense(hidden_size -> 256) -> ReLU -> Dense(256 -> num_classes)
Loss function:          Hierarchical Cross-Entropy (HCE) - CORRECTED
Optimizer:              AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})
Epochs:                 {N_EPOCHS}
Batch size:             {BATCH_SIZE}

KEY CORRECTIONS IN THIS VERSION
================================
✅ Reachability matrix directionality: Parent→Child (was backwards)
✅ Transitive closure: Floyd-Warshall algorithm applied
✅ HCE loss formula: Uses matrix[:, true_label] for ancestor selection
✅ All {torch.count_nonzero(reachability_matrix).item() - len(le.classes_)} parent-child relationships now included

TRAINING RESULTS
================
Final Train Loss:       {train_losses[-1]:.4f}
Final Val Loss:         {val_losses[-1]:.4f}
Final Val Accuracy:     {val_accs[-1]:.4f}

TEST SET PERFORMANCE
====================
Test Loss:              {test_loss:.4f}
Test Accuracy:          {test_acc:.4f}

Top 5 Cell Types by F1 Score:
"""

for idx, row in per_class_metrics.head(5).iterrows():
    summary_text += f"  {row['cell_type']:40s}: F1={row['f1']:.4f} (P={row['precision']:.4f}, R={row['recall']:.4f})\n"

summary_text += f"""
KEY ADVANTAGES OF CORRECTED HCE LOSS
====================================
• Hierarchy is correctly incorporated into the training objective
• All {torch.count_nonzero(reachability_matrix).item() - len(le.classes_)} hierarchical relationships are now active
• Encoder learns task-specific representations optimized for hierarchical classification
• Partial credit given ONLY for ancestor predictions (correct direction)
• Transitive closure ensures multi-level hierarchy is properly represented
• End-to-end optimization of encoder + classifier
• Better generalization by leveraging hierarchical structure correctly

OUTPUT FILES
============
• training_curves.png - Loss and accuracy during training
• confusion_matrix.png - Normalized confusion matrix heatmap
• per_class_metrics.png - Precision/Recall/F1 for top cell types
• per_class_metrics.csv - Complete per-class metrics
• ontology.csv - Cell type hierarchy
• balanced_counts.csv - Class distribution after balancing
"""

print(summary_text)

# Save summary
summary_path = os.path.join(OUT_DIR, "hce_training_summary.txt")
with open(summary_path, 'w') as f:
    f.write(summary_text)

print(f"\n✅ Summary saved to {summary_path}")
print(f"\n✅ All outputs saved to: {OUT_DIR}")